<a href="https://colab.research.google.com/github/KasunUdayanga/NER-Sinhala-political-comment-identifier/blob/main/xmlr-large.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Create Label-to-ID Mapping

### Subtask:
Identify all unique NER tags present in the dataset and create a mapping from each tag string to a unique integer ID, and a reverse mapping from ID back to tag. This is necessary for numerical processing by the model.


**Reasoning**:
I need to initialize an empty set to store unique NER tags, iterate through the `conll_data` to populate this set, then create `tag_to_id` and `id_to_tag` dictionaries, and finally print the results.



In [16]:
unique_tags = set()

for entry in conll_data:
    for label in entry['labels']:
        unique_tags.add(label)

# Convert set to a sorted list to ensure consistent ID assignment
sorted_unique_tags = sorted(list(unique_tags))

# Create tag_to_id mapping
tag_to_id = {tag: i for i, tag in enumerate(sorted_unique_tags)}

# Create id_to_tag mapping
id_to_tag = {i: tag for i, tag in enumerate(sorted_unique_tags)}

print(f"Total unique NER tags found: {len(unique_tags)}")
print("\nTag to ID mapping:")
print(tag_to_id)
print("\nID to Tag mapping:")
print(id_to_tag)


Total unique NER tags found: 9

Tag to ID mapping:
{'B-LOC': 0, 'B-ORG': 1, 'B-Other': 2, 'B-PER': 3, 'I-LOC': 4, 'I-ORG': 5, 'I-Other': 6, 'I-PER': 7, 'O': 8}

ID to Tag mapping:
{0: 'B-LOC', 1: 'B-ORG', 2: 'B-Other', 3: 'B-PER', 4: 'I-LOC', 5: 'I-ORG', 6: 'I-Other', 7: 'I-PER', 8: 'O'}


## Tokenize Sentences and Align Labels

### Subtask:
For each sentence, tokenize the words using the `bert-base-multilingual-cased` tokenizer. Then, align the NER tags with the generated subword tokens. This involves handling cases where a single word splits into multiple subwords, ensuring that each subword token gets an appropriate label (e.g., using -100 for padding tokens or subsequent subwords that should be ignored in loss calculation).


**Reasoning**:
I need to import the `AutoTokenizer` from the `transformers` library, initialize it, define the `tokenize_and_align_labels` function as specified in the instructions, and then apply this function to each sentence in `conll_data` to get tokenized input IDs and aligned labels.



In [17]:
from transformers import AutoTokenizer

# 2. Initialize a tokenizer
model_name = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align_labels(tokens, labels, tokenizer, tag_to_id):
    # 4. Tokenize the tokens
    tokenized_inputs = tokenizer(tokens, is_split_into_words=True, truncation=True)

    # 5. Get the word_ids
    word_ids = tokenized_inputs.word_ids()

    aligned_labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None: # 7.1. Special tokens or padding
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx: # 7.2. First subword token of an original word
            # Get the label for the current original word
            original_label = labels[word_idx]
            aligned_labels.append(tag_to_id[original_label])
        else: # 7.3. Subsequent subword token of the same original word
            aligned_labels.append(-100)
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = aligned_labels
    tokenized_inputs["decoded_tokens"] = tokenizer.convert_ids_to_tokens(tokenized_inputs['input_ids']) # Add decoded tokens
    return tokenized_inputs

tokenized_and_aligned_data = []
print(f"Tokenizing and aligning labels for {len(conll_data)} sentences...")
for i, entry in enumerate(conll_data):
    tokenized_output = tokenize_and_align_labels(entry['tokens'], entry['labels'], tokenizer, tag_to_id)
    tokenized_and_aligned_data.append(tokenized_output)
    if i < 5: # Print first 5 for sample
        print(f"\nSample sentence {i+1}:")
        print(f"  Original Tokens: {entry['tokens']}")
        print(f"  Original Labels: {entry['labels']}")
        print(f"  Input IDs: {tokenized_output['input_ids']}")
        print(f"  Aligned Labels: {tokenized_output['labels']}")
        print(f"  Decoded Tokens: {tokenized_output['decoded_tokens']}")

if len(conll_data) > 5:
    print(f"\n... and {len(conll_data) - 5} more tokenized and aligned sentences.")

print(f"Finished tokenizing and aligning labels for {len(tokenized_and_aligned_data)} sentences.")

Tokenizing and aligning labels for 9633 sentences...

Sample sentence 1:
  Original Tokens: ['පාලමුන']
  Original Labels: ['B-LOC']
  Input IDs: [0, 237394, 80078, 2]
  Aligned Labels: [-100, 0, -100, -100]
  Decoded Tokens: ['<s>', '▁පාලම', 'ුන', '</s>']

Sample sentence 2:
  Original Tokens: ['ඔබතමයි', 'කාරයෝන්ට', 'රාජපක්ෂලාටම්', 'හර්ශ', 'විපක්ශයට', 'අවලද', 'දැම්මාඇත්ත', 'ඩොබිලාට', 'පැන්චො']
  Original Labels: ['B-Other', 'B-Other', 'B-PER', 'B-PER', 'B-ORG', 'B-Other', 'B-Other', 'B-Other', 'B-Other']
  Input IDs: [0, 11838, 84263, 2009, 6, 25444, 19687, 24625, 52374, 3998, 722, 4490, 14235, 68725, 210630, 27436, 6733, 28264, 2885, 1859, 225608, 72980, 30498, 6, 146221, 49485, 3998, 722, 19853, 2148, 29057, 25859, 2]
  Aligned Labels: [-100, 2, -100, -100, 2, -100, -100, -100, 3, -100, -100, -100, 3, -100, 1, -100, -100, 2, -100, -100, 2, -100, -100, 2, -100, -100, -100, -100, 2, -100, -100, -100, -100]
  Decoded Tokens: ['<s>', '▁ඔබ', 'තම', 'යි', '▁', 'කාර', 'යෝ', 'න්ට', '▁රාජපක්ෂ'

# Task
Split the `tokenized_and_aligned_data` into training and testing sets, then convert these sets into `torch.utils.data.Dataset` and `torch.utils.data.DataLoader` objects to prepare the data for model training and evaluation.

## Split Data into Train and Test Sets

### Subtask:
Divide the `tokenized_and_aligned_data` into training and testing sets to evaluate the model's generalization capabilities. A typical split would be 80% for training and 20% for testing.


**Reasoning**:
I need to import the `train_test_split` function from `sklearn.model_selection` to split the dataset.



In [18]:
from sklearn.model_selection import train_test_split

**Reasoning**:
I need to split the `tokenized_and_aligned_data` into training and testing sets using an 80/20 ratio and then print the sizes of the resulting sets, as instructed by the subtask.



In [19]:
train_data, test_data = train_test_split(tokenized_and_aligned_data, test_size=0.2, random_state=42)

print(f"Number of samples in training set: {len(train_data)}")
print(f"Number of samples in testing set: {len(test_data)}")

Number of samples in training set: 7706
Number of samples in testing set: 1927


## Create PyTorch Dataset and DataLoader

### Subtask:
Convert the tokenized and aligned training and test data into `torch.utils.data.Dataset` objects and then create `torch.utils.data.DataLoader` instances for efficient batching during training and evaluation.


**Reasoning**:
First, I need to import the necessary PyTorch modules, `Dataset` and `DataLoader`, to define and use custom datasets and data loaders, which is the foundational step for preparing data for model training.



In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

print("Imported torch Dataset and DataLoader.")

Imported torch Dataset and DataLoader.


**Reasoning**:
I need to define a custom PyTorch Dataset class to properly handle the tokenized and aligned data for training, as instructed by the subtask.



In [21]:
class NERDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]
        return {
            'input_ids': torch.tensor(entry['input_ids'], dtype=torch.long),
            'attention_mask': torch.tensor(entry['attention_mask'], dtype=torch.long),
            'labels': torch.tensor(entry['labels'], dtype=torch.long)
        }

print("Defined custom NERDataset class.")

Defined custom NERDataset class.


**Reasoning**:
Now that the custom `NERDataset` class is defined, I need to create instances of this dataset for both the training and testing data, and then initialize `DataLoader` instances for each, setting the appropriate batch size and shuffling for training, as per the subtask instructions.



In [22]:
batch_size = 16 # You can adjust this value

# Create Dataset objects
train_dataset = NERDataset(train_data)
test_dataset = NERDataset(test_data)

# Create DataLoader objects
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train Dataset size: {len(train_dataset)}")
print(f"Test Dataset size: {len(test_dataset)}")
print(f"Train DataLoader created with batch size: {batch_size}")
print(f"Test DataLoader created with batch size: {batch_size}")

Train Dataset size: 7706
Test Dataset size: 1927
Train DataLoader created with batch size: 16
Test DataLoader created with batch size: 16


## Load XLM-RoBERTa Model for Token Classification

### Subtask:
Load the pre-trained `xlm-roberta-large` model with a token classification head, specifically `XLMRobertaForTokenClassification`. The model should be initialized with the number of unique NER tags (num_labels) identified earlier.


**Reasoning**:
I need to import the `XLMRobertaForTokenClassification` class, calculate `num_labels` from `id_to_tag`, and then load the pre-trained model with the specified `model_name` and `num_labels` to fulfill the subtask.



In [23]:
from transformers import XLMRobertaForTokenClassification

# Determine the number of unique NER tags
num_labels = len(id_to_tag)

# Load the pre-trained XLMRobertaForTokenClassification model
model = XLMRobertaForTokenClassification.from_pretrained(model_name, num_labels=num_labels)

print(f"Loaded {model_name} model with {num_labels} labels for token classification.")
print(f"Model configuration: {model.config}")

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded xlm-roberta-large model with 9 labels for token classification.
Model configuration: XLMRobertaConfig {
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8"
  },
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6,
    "LABEL_7": 7,
    "LABEL_8": 8
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "output_past": true,
  "pad_token_i

**Reasoning**:
The previous execution produced a warning about uninitialized weights in the classifier head, which is expected behavior when loading a pre-trained model for a new token classification task. The code itself is correct and performs the requested action of loading the model with the specified number of labels. No code changes are required to address this warning.



In [24]:
from transformers import XLMRobertaForTokenClassification

# Determine the number of unique NER tags
num_labels = len(id_to_tag)

# Load the pre-trained XLMRobertaForTokenClassification model
model = XLMRobertaForTokenClassification.from_pretrained(model_name, num_labels=num_labels)

print(f"Loaded {model_name} model with {num_labels} labels for token classification.")
print(f"Model configuration: {model.config}")

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loaded xlm-roberta-large model with 9 labels for token classification.
Model configuration: XLMRobertaConfig {
  "architectures": [
    "XLMRobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8"
  },
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6,
    "LABEL_7": 7,
    "LABEL_8": 8
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "xlm-roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "output_past": true,
  "pad_token_i

**Reasoning**:
The model has been successfully loaded. The next logical step in the process of training a Hugging Face model is to define the training arguments, which includes parameters like output directory, learning rate, number of epochs, etc. This prepares the model for the training loop.



In [31]:
from transformers import TrainingArguments
from transformers.trainer_utils import EvaluationStrategy

# Define training arguments
training_args = TrainingArguments(
    output_dir="./ner_model_output",  # Output directory for model checkpoints and logs
    eval_strategy="epoch",    # Evaluate every epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=16, # Batch size per device during training
    per_device_eval_batch_size=16,  # Batch size per device during evaluation
    num_train_epochs=3,             # Total number of training epochs
    weight_decay=0.01,              # Strength of weight decay
    logging_dir='./logs',           # Directory for storing logs
    logging_steps=100,              # Log every 100 steps
    save_strategy="epoch",          # Save model checkpoint every epoch
    load_best_model_at_end=True,    # Load the best model at the end of training
    metric_for_best_model="f1",     # Metric to use to compare models
    push_to_hub=False               # Whether or not to push to the Hugging Face Model Hub
)

print("TrainingArguments initialized.")

TrainingArguments initialized.


## Define Evaluation Metrics Function

### Subtask:
Create a function to compute standard NER evaluation metrics (precision, recall, F1-score) using the `seqeval` library. This function will correctly handle the `-100` padding in the labels.


**Reasoning**:
I need to import the `evaluate` library and `numpy` for numerical operations, and then load the `seqeval` metric to prepare for defining the `compute_metrics` function, as instructed by the subtask.



In [33]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00


**Reasoning**:
The `evaluate` library has been successfully installed. Now, I can proceed with importing it along with `numpy` and loading the `seqeval` metric, as originally intended in the subtask instructions.



In [35]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=91b49ce7f5ad8ee89c743b47490dc55487a059f19316f0e2e6928b9e50efe390
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


**Reasoning**:
Now that `seqeval` is installed, I will re-attempt to import `evaluate` and `numpy`, and then load the `seqeval` metric, as this was the original intent before encountering the import error.



In [36]:
import evaluate
import numpy as np

# Load the 'seqeval' metric
metric = evaluate.load('seqeval')

print("Imported 'evaluate' and 'numpy', and loaded 'seqeval' metric.")

Imported 'evaluate' and 'numpy', and loaded 'seqeval' metric.


**Reasoning**:
Now that the necessary libraries and metric are loaded, I need to define the `compute_metrics` function as instructed, which will take model predictions and true labels, process them to remove padding, and then compute and return the NER evaluation metrics using `seqeval`.



In [37]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (padding) and convert to string labels
    true_labels = [
        [id_to_tag[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [id_to_tag[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "overall_precision": results["overall_precision"],
        "overall_recall": results["overall_recall"],
        "overall_f1": results["overall_f1"],
        "overall_accuracy": results["overall_accuracy"],
    }

print("Defined compute_metrics function.")

Defined compute_metrics function.


## Initialize and Train the Trainer

### Subtask:
Initialize the Hugging Face `Trainer` with the model, training arguments, data loaders, and the compute metrics function. Then, start the fine-tuning process on the training data.


**Reasoning**:
I need to import the `Trainer` class from the `transformers` library, which is the first instruction to initialize the Hugging Face `Trainer`.



In [38]:
from transformers import Trainer

print("Imported Trainer class.")

Imported Trainer class.


**Reasoning**:
Now that the `Trainer` class is imported, I need to initialize a `Trainer` object with the model, training arguments, datasets, and evaluation metrics function, and then start the fine-tuning process as per the subtask instructions.



In [40]:
class NERDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        entry = self.data[idx]
        return {
            'input_ids': entry['input_ids'],
            'attention_mask': entry['attention_mask'],
            'labels': entry['labels']
        }

print("Defined custom NERDataset class to return raw lists for proper batching.")

Defined custom NERDataset class to return raw lists for proper batching.


**Reasoning**:
The `NERDataset` class has been redefined. I need to recreate the `train_dataset` and `test_dataset` instances using the corrected `NERDataset` definition, and then re-initialize the `DataLoader` objects to reflect these changes. This ensures that the `Trainer` will use the properly formatted data.



In [41]:
batch_size = 16 # You can adjust this value

# Create Dataset objects with the corrected NERDataset class
train_dataset = NERDataset(train_data)
test_dataset = NERDataset(test_data)

# Create DataLoader objects
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train Dataset size: {len(train_dataset)}")
print(f"Test Dataset size: {len(test_dataset)}")
print(f"Train DataLoader created with batch size: {batch_size}")
print(f"Test DataLoader created with batch size: {batch_size}")

Train Dataset size: 7706
Test Dataset size: 1927
Train DataLoader created with batch size: 16
Test DataLoader created with batch size: 16


**Reasoning**:
The previous `ValueError` indicates that the `Trainer`'s internal data collator is failing to pad sequences of varying lengths before converting them to tensors. While passing `tokenizer` to the `Trainer` should implicitly use `DataCollatorForTokenClassification`, explicitly defining and passing an instance of `DataCollatorForTokenClassification` with appropriate padding parameters will ensure that the data is correctly padded for all `input_ids`, `attention_mask`, and `labels` before tensor conversion, thereby resolving the `ValueError`.



In [43]:
from transformers import DataCollatorForTokenClassification, Trainer

# Explicitly define the data collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, label_pad_token_id=-100)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator # Pass the explicitly created data collator
)

print("Trainer initialized. Starting training...")
trainer.train()
print("Training completed.")

Trainer initialized. Starting training...


Epoch,Training Loss,Validation Loss,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,0.962000,0.811775,0.672803,0.879062,0.762225,0.676472


KeyError: "The `metric_for_best_model` training argument is set to 'eval_f1', which is not found in the evaluation metrics. The available evaluation metrics are: ['eval_loss', 'eval_overall_precision', 'eval_overall_recall', 'eval_overall_f1', 'eval_overall_accuracy']. Consider changing the `metric_for_best_model` via the TrainingArguments."

**Reasoning**:
The previous error occurred because the `metric_for_best_model` in `TrainingArguments` was set to `'f1'`, but the `compute_metrics` function returns `'overall_f1'`. To resolve this, I need to update the `metric_for_best_model` in the `training_args` to `'overall_f1'` to match the actual metric returned.



In [44]:
from transformers import TrainingArguments
from transformers.trainer_utils import EvaluationStrategy

# Define training arguments
training_args = TrainingArguments(
    output_dir="./ner_model_output",  # Output directory for model checkpoints and logs
    eval_strategy="epoch",    # Evaluate every epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=16, # Batch size per device during training
    per_device_eval_batch_size=16,  # Batch size per device during evaluation
    num_train_epochs=3,             # Total number of training epochs
    weight_decay=0.01,              # Strength of weight decay
    logging_dir='./logs',           # Directory for storing logs
    logging_steps=100,              # Log every 100 steps
    save_strategy="epoch",          # Save model checkpoint every epoch
    load_best_model_at_end=True,    # Load the best model at the end of training
    metric_for_best_model="overall_f1",     # Metric to use to compare models
    push_to_hub=False               # Whether or not to push to the Hugging Face Model Hub
)

print("TrainingArguments initialized.")

TrainingArguments initialized.


**Reasoning**:
The previous training attempt resulted in an `OutOfMemoryError` on the GPU, indicating that the current batch size is too large for the available memory. To fix this, I need to reduce the `per_device_train_batch_size` and `per_device_eval_batch_size` in the `TrainingArguments` to decrease memory consumption and allow the model to train.



In [46]:
from transformers import TrainingArguments
from transformers.trainer_utils import EvaluationStrategy

# Define training arguments
training_args = TrainingArguments(
    output_dir="./ner_model_output",  # Output directory for model checkpoints and logs
    eval_strategy="epoch",    # Evaluate every epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=8, # Reduced batch size per device during training
    per_device_eval_batch_size=8,  # Reduced batch size per device during evaluation
    num_train_epochs=3,             # Total number of training epochs
    weight_decay=0.01,              # Strength of weight decay
    logging_dir='./logs',           # Directory for storing logs
    logging_steps=100,              # Log every 100 steps
    save_strategy="epoch",          # Save model checkpoint every epoch
    load_best_model_at_end=True,    # Load the best model at the end of training
    metric_for_best_model="overall_f1",     # Metric to use to compare models
    push_to_hub=False               # Whether or not to push to the Hugging Face Model Hub
)

print("TrainingArguments initialized with reduced batch size.")

TrainingArguments initialized with reduced batch size.


**Reasoning**:
Now that the `TrainingArguments` have been updated with a reduced batch size, I need to re-initialize the `Trainer` object with these new arguments and then restart the fine-tuning process. This should resolve the `OutOfMemoryError` encountered previously and allow the training to proceed as intended.



In [ ]:
from transformers import DataCollatorForTokenClassification, Trainer

# Explicitly define the data collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer, padding=True, label_pad_token_id=-100)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator # Pass the explicitly created data collator
)

print("Trainer initialized with reduced batch size. Starting training...")
trainer.train()
print("Training completed.")

Trainer initialized with reduced batch size. Starting training...


Epoch,Training Loss,Validation Loss
